In [ ]:
from icw.auto_watermark import AutoWatermark
from datasets import load_dataset
from tqdm import tqdm
from evaluation.evaluation import Evaluation
import pandas as pd

In [ ]:
import os

EXTRA_INSTRUCTION = " "

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')


unicode_config = {
    'icw': 'UNICODE',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for high reasoning mode; 'n.n' for non-reasoning
    'api_key': OPENAI_API_KEY,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

lexical_config = {
    'icw': 'LEXICAL',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for reasoning; 'n.n' for non-reasoning
    'api_key': OPENAI_API_KEY,
    'seed': 123,
    'gamma': 0.20,
    'r_high_freq': 300,  
    'r_low_freq': 13000,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

initials_config = {
    'icw': 'INITIALS',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for reasoning; 'n.n' for non-reasoning
    'api_key': OPENAI_API_KEY,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

acrostics_config = {
    'icw': 'ACROSTICS',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for reasoning; 'n.n' for non-reasoning
    'seed': 123,
    'str_len': 20,
    'api_key': OPENAI_API_KEY,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

eva_config = {
    'model': 'gpt-4o-mini',
    'api_key': OPENAI_API_KEY,
}

In [ ]:
unicodeWatermark = AutoWatermark.load(unicode_config)

In [ ]:
lexicalWatermark = AutoWatermark.load(lexical_config)

In [ ]:
initialsWatermark = AutoWatermark.load(initials_config)

In [ ]:
acrosticsWatermark = AutoWatermark.load(acrostics_config)

In [ ]:
evaluation = Evaluation(eva_config)

### Indirect Prompt Inject (IPI) Setting

In [ ]:
ds_paper = load_dataset("WestlakeNLP/Research-14K", split="test")
ds_review = load_dataset("WestlakeNLP/Review-5K", split="test")

# icw = unicodeWatermark
# icw = lexicalWatermark
icw = initialsWatermark
# icw = acrosticsWatermark

rows = []
for i in tqdm(range(5)):
    _id = i
    paper = str(ds_paper[i]['sections'])
    reference_review  = str(ds_review[i]['review_contexts'])

    wm_text = icw.indirect_prompt_injection(paper)
    wm_score = icw.detect_watermark(wm_text)
    unwm_score = icw.detect_watermark(reference_review)

    # p_wm_text = evaluation.paraphrase(wm_text)
    # p_wm_score = icw.detect_watermark(p_wm_text)

    rows.append({
        'id': _id,
        'paper': paper,
        'reference_review': reference_review,
        'wm_text': wm_text,
        # 'p_wm_text': p_wm_text,
        'wm_score': wm_score,
        'unwm_score': unwm_score,
        # 'p_wm_score': p_wm_score
    })

    df = pd.DataFrame(rows)
    df.to_csv('outputs/ipi/' + type(icw).__name__ + '_ipi.csv', index=False)